In [16]:
from pathlib import Path
from pydub import AudioSegment
from config import config

In [17]:
root = Path("../../..")

In [18]:
SOURCE_DIR = root / "data/audio/bn"
OUTPUT_DIR = root / config.RAW_AUDIO_DIR_BN
METADATA_FILE = root / config.METADATA_DIR / "bn/metadata_bn.csv"
CLIP_DURATION_MS = 3000 # 3 seconds

In [19]:
SOURCE_DIR.exists(), METADATA_FILE.exists() # must be true

(True, True)

In [20]:
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import shutil

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

In [21]:
import math, pandas as pd

In [22]:
N_SUBSET = math.ceil(config.N_SUBSET / 2)

In [23]:
# if n_subset no. of clips are to be considered, then roughly n_subset/10 no. of audio files should be included. to ensure the quadrant distribution of the audio files remain balanced, these n_subset/10 audio files can be sampled randomly first and then their corresponding features extracted??
num_audios = math.ceil(N_SUBSET / 10)

In [24]:
metadata_df = pd.read_csv(METADATA_FILE)

In [27]:
sampled_df = metadata_df.sample(n=num_audios, random_state=43).reset_index(drop=True)

In [28]:
sampled_df.head()

,id,sentence,genre
0,-ph4mykFp9I_seg_00005,সাগরে মানুষের হৃদয় ভরে ওঠে আবার শূন্য হবার জন...,Alternative
1,ECh1rS2ipJw_seg_00003,বাজিয়ে আমার মনের মত দিও সাজিয়ে আমি বড় অসহায...,Rock
2,KdURvuVydnI_seg_00001,আমি নিতে দেবো না সময়কে এক মুঠো ভরা যোচনা যাই ...,Alternative
3,mli5roiIW-s_seg_00010,সকাল আয় ঘুম চুম বন্দে তার সারা কপালে যাতে ঘুম...,Indie/Folk-Pop
4,nPmCDRMwvaA_seg_00004,দেহের বাইরে প্রেম আমার সত্য আত্মা কঠিন কঠিন সব...,Rock


In [29]:
audio_file_list = sampled_df["id"].to_list()

In [30]:
len(audio_file_list)

125

In [31]:
audio_file_list[:5]

['-ph4mykFp9I_seg_00005',
 'ECh1rS2ipJw_seg_00003',
 'KdURvuVydnI_seg_00001',
 'mli5roiIW-s_seg_00010',
 'nPmCDRMwvaA_seg_00004']

In [32]:
missing = 0
for audio_filename in audio_file_list:
    audio_file = [file for file in SOURCE_DIR.iterdir() if file.is_file() and file.stem == audio_filename][0]
    try:
        if not audio_file: 
            missing += 1
            continue
        audio = AudioSegment.from_file(audio_file)
        total_length = len(audio)
        n_clips = total_length // CLIP_DURATION_MS
        for i in range(n_clips):
            start = i * CLIP_DURATION_MS
            end = start + CLIP_DURATION_MS
            clip = audio[start:end]
            output_file_name = f"{audio_file.stem}_clip_{i+1}.wav"
            output_file_path = OUTPUT_DIR / output_file_name
            clip.export(output_file_path, format="wav")
    except Exception as e:
        print(e)

In [33]:
missing

0

In [34]:
sampled_df.to_csv(METADATA_FILE, index=False)